In [1]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

def read_json(file_name):
    # with open(file_name, 'r') as file:
        # return [json.loads(line) for line in file]
    with open(file_name, 'r') as file:
        return json.load(file)
def json_dataset_parser(jsons_list, labels_dict):
    data_dict = {"text": [], "labels": [], "domain": []}
    for obj in jsons_list:
        data_dict["text"].append(obj["text"])
        data_dict["labels"].append(labels_dict[obj["label"]])
        data_dict["domain"].append(obj["domain"])
    return pd.DataFrame(data_dict)

def prepare_dataset(file_path, labels_dict, test_size=0.15, val_size=0.15, sample_frac=1.0):
    jsons_list = read_json(file_path)
    df = json_dataset_parser(jsons_list, labels_dict)
    df = df.sample(frac=sample_frac).reset_index(drop=True)

    train_val, test = train_test_split(df, test_size=test_size, stratify=df['labels'])
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), stratify=train_val['labels'])

    dataset = DatasetDict({
        'train': Dataset.from_pandas(train),
        'val': Dataset.from_pandas(val),
        'test': Dataset.from_pandas(test)
    })
    return dataset

file_path = '/kaggle/input/data-correct/data_correct.json'
labels_dict = {
        "human_text": 0,
        "machine_text": 1
    }
  

# data = prepare_dataset(file_path, labels_dict)

data = read_json(file_path)

data[11]

texts = [d['text'] for d in data]

len(texts)

import torch
from tqdm import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
def calculate_ppl(model, tokenizer, stride, max_length, device=device):
  model.eval()

  ppl = []
  for text in tqdm(texts):
      encodings = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

      seq_len = encodings.input_ids.size(1)

      nlls = []
      prev_end_loc = 0
      for begin_loc in range(0, seq_len, stride):
          end_loc = min(begin_loc + max_length, seq_len)
          trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
          input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
          target_ids = input_ids.clone()
          target_ids[:, :-trg_len] = -100

          with torch.no_grad():
              outputs = model(input_ids, labels=target_ids)

              # loss is calculated using CrossEntropyLoss which averages over valid labels
              # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
              # to the left by 1.
              neg_log_likelihood = outputs.loss

          nlls.append(neg_log_likelihood)

          prev_end_loc = end_loc
          if end_loc == seq_len:
              break

      ppl.append(torch.exp(torch.stack(nlls).mean()))
  return ppl

In [2]:
!pip install -U bitsandbytes
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.10.19
    Uninstalling nvidia-curand-cu12-10

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)
tokenizer_llama = AutoTokenizer.from_pretrained("huihui-ai/Llama-3.2-3B-Instruct-abliterated")
model_llama = AutoModelForCausalLM.from_pretrained("huihui-ai/Llama-3.2-3B-Instruct-abliterated",
                                                   quantization_config=bnb_config)

tokenizer_config.json:   0%|          | 0.00/54.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

2025-05-13 12:17:22.723120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747138642.945455      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747138643.009234      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.25G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [4]:
ppl_llama3 = calculate_ppl(model_llama, tokenizer_llama, 256, 512)

100%|██████████| 61797/61797 [4:08:58<00:00,  4.14it/s]


In [5]:
import torch
import numpy as np
ppl_llama3 = torch.Tensor(ppl_llama3)
np.save('/kaggle/working/ppl_llama3.npy', ppl_llama3)